# SageAgent V4.10.9

**Setup:** Run cells 1-3 in order. Cell 1 installs packages (once). Cell 2 shows config widgets. Cell 3 launches the agent.

**Core files:** `sagemaker_agent.py` (the entire agent) + this notebook + `memory.md` (auto-populated) + `AGENT_STATUS.md` (long-running handoff) + `skills/` (10 skills + `/reflexion`).

**v4.9 highlights** (full notes in `changelogs/CHANGELOG_v4.9.X.md`):
- v4.9.0 — `auto_trigger: false` honoured; word-boundary keyword match; SYSTEM_PROMPT critique-handling section
- v4.9.1 — `/unskill <name>` + sticky deactivation
- v4.9.2 — minimum-ship zip
- v4.9.3 — prompt-injection scanner; CSO description validator; `/reflexion` skill
- v4.9.4 — IterationBudget shared parent+sub-agents; ErrorClassifier + jittered backoff; pre-compact tool-result pruning; opt-in auxiliary-model compaction (Haiku); structured Resolved/Pending Questions in summary
- v4.9.5 — opt-in self-patching skills (agent proposes SKILL.md improvements, you review via `/skill suggestions` then `/skill apply <name>`)
- v4.9.6 — production hardening: no default skill auto-load, Bedrock-safe compact history, fixed build worktree isolation, dirty-state worktree overlay, serialized parallel builds, durable status doc, local-git-only SageMaker policy
- v4.9.7 — `/context` diagnostic: context percentage, top tool-output sources, duplicate file reads, suggested action
- v4.10.0 — Runnable parity (5 phases): notebook_edit (surgical .ipynb cell editor), skill listing token budget cap, per-sub-agent env-details, model-aware context window auto-derive, reactive compact on Bedrock CONTEXT_OVERFLOW. 5 Codex per-phase reviews, 9 correctness issues caught and fixed.
- v4.10.1 — same-day follow-up: segment-level Context Collapse (collapses 3+ stale tool round-trips into one synthetic Bedrock-safe pair) + default model switched to Sonnet 4.5 (cache activates from 1024 tokens vs Haiku 4096).
- v4.10.2 — verify-contract softened: now SUGGEST /verify after 3+ logic edits and wait for user confirm; strict mode opt-in via CONFIG.enforce_verify_contract=True (Codex caught the prompt contradiction)
- v4.10.3 — production-readiness review apply: ship-gate verifier (verify_ship_zip.py), cache-boundary regression test (6 tests), many-skill stress test (100/1000 skills), clearer permission denials
- v4.10.4 — sub-agent work-context handoff (bounded AGENT_STATUS slice + active todos + last 10 changed files), Codex follow-up review fix
- v4.10.5 — Learning_Factory pattern adoption: post-compact resume protocol + structured summary sections (#12 Standing Constraints + #13 Critical Don't-Forget) + skill self-patching 4-rule check (Repeated + Non-trivial + Generalizable + Real-pitfall)
- v4.10.6 — html skill: presentation / design / flowchart / architecture HTML deliverables. Ships 3 reference templates (tabbed_design, presentation_slides, flowchart_page). Screenshot-iteration loop for SageMaker (no Playwright).
- v4.10.7 — destructive-command hardening: ~50 new bash DANGEROUS_PATTERNS + ~12 new python DANGEROUS_PYTHON patterns. HIGH_RISK_TOOLS = {bash, python_exec, task, web_fetch} excluded from Always-Approve UI. 107-case test coverage. Cross-surface mirrored into Claude Code + Learning_Factory hooks.
- v4.10.8 — obfuscation hardening + recursive folder removal hard-block. Bash: base64-decode pipe to extended interpreter set, xxd/od/hexdump decode chains, recursive `rm`/`rmdir`/PowerShell `Remove-Item -Recurse`. Python: blanket `shutil.rmtree`/`os.rmdir`/`os.removedirs`/`Path.rmdir` block from `python_exec`. 129-case test coverage.
- **v4.10.9** — backtick eval+downloader parity. One narrow new pattern catching `eval` + backtick + remote-fetcher (`curl`/`wget`/`fetch`) or decoder (`base64`/`xxd`/`hexdump`) — closes Codex's third local-hook finding from v4.10.8 round in v4 itself. Defense-in-depth only — `eval` is already excluded from v4's bash allowlist, zero new false-positive risk. 134-case test coverage (up from 129).

**Full docs:** See `USER_GUIDE.md` for complete documentation.


In [ ]:
# Install dependencies (run once)
!pip install -q boto3 ipywidgets Pillow python-docx pandas openpyxl

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
# Models are imported from sagemaker_agent.py (single source of truth)

import ipywidgets as widgets
from IPython.display import display, HTML
from sagemaker_agent import BEDROCK_MODELS

# Convert BEDROCK_MODELS list-of-tuples to dict for config cell
AVAILABLE_MODELS = dict(BEDROCK_MODELS)

# Temperature options
TEMPERATURE_OPTIONS = {
    "0.0 - Deterministic": 0.0,
    "0.3 - Low creativity": 0.3,
    "0.5 - Balanced": 0.5,
    "0.7 - High creativity": 0.7,
    "1.0 - Maximum creativity": 1.0,
}

# Thinking budget options
THINKING_BUDGET_OPTIONS = {
    "1024 - Minimal": 1024,
    "2048 - Light": 2048,
    "4096 - Standard": 4096,
    "8192 - Extended": 8192,
    "16000 - Maximum": 16000,
}

# Region - Sydney (ap-southeast-2)
REGION = "ap-southeast-2"

# Create configuration widgets
display(HTML("<h3>Agent Configuration</h3>"))

# Use first model as default (matches BEDROCK_MODELS order)
default_model_name = list(AVAILABLE_MODELS.keys())[0]

model_dropdown = widgets.Dropdown(
    options=list(AVAILABLE_MODELS.keys()),
    value=default_model_name,
    description='Model:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='450px')
)

temperature_dropdown = widgets.Dropdown(
    options=list(TEMPERATURE_OPTIONS.keys()),
    value="0.0 - Deterministic",
    description='Temperature:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='350px')
)

thinking_checkbox = widgets.Checkbox(
    value=False,
    description='Enable Extended Thinking (slower, uses more tokens)',
    indent=False,
    style={'description_width': 'auto'}
)

thinking_budget_dropdown = widgets.Dropdown(
    options=list(THINKING_BUDGET_OPTIONS.keys()),
    value="4096 - Standard",
    description='Thinking Budget:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='350px')
)

max_turns_slider = widgets.IntSlider(
    value=60,
    min=5,
    max=100,
    step=5,
    description='Max Turns:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

workspace_input = widgets.Text(
    value='.',
    description='Workspace:',
    placeholder='Directory for file operations',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

mock_toggle = widgets.Checkbox(
    value=False,
    description='Mock Mode (test without API)',
    indent=False,
    style={'description_width': 'auto'}
)

# Display configuration UI
config_box = widgets.VBox([
    model_dropdown,
    widgets.HTML(f"<p style='margin:5px 0;color:#888;'>Region: Sydney ({REGION})</p>"),
    temperature_dropdown,
    thinking_checkbox,
    thinking_budget_dropdown,
    workspace_input,
    max_turns_slider,
    mock_toggle,
], layout=widgets.Layout(padding='10px', border='1px solid #444', margin='10px 0', background='#2d2d2d'))

display(config_box)
display(HTML("<p style='color:#888;font-size:12px;'>Configure settings above, then run the next cell to start.</p>"))

In [ ]:
# ============================================================
# LAUNCH AGENT WITH CONFIGURATION
# ============================================================

from sagemaker_agent import CONFIG, create_chat_ui
from IPython.display import display, HTML

# Security settings
CONFIG.aws_bedrock_only = True          # Block ALL AWS services except Bedrock
CONFIG.require_tool_approval = True     # Show Approve/Deny dialog before execution
CONFIG.session_cost_limit = 5.0         # Default $5 — adjustable via Budget slider in UI

# Apply configuration from widgets above
CONFIG.model_id = AVAILABLE_MODELS[model_dropdown.value]
CONFIG.region = REGION  # Sydney
CONFIG.workspace = workspace_input.value
CONFIG.max_turns = max_turns_slider.value
CONFIG.mock_mode = mock_toggle.value
CONFIG.temperature = TEMPERATURE_OPTIONS[temperature_dropdown.value]
CONFIG.thinking_enabled = thinking_checkbox.value
CONFIG.thinking_budget = THINKING_BUDGET_OPTIONS[thinking_budget_dropdown.value]

# Display current config
thinking_str = f"Thinking: On (budget: {CONFIG.thinking_budget})" if CONFIG.thinking_enabled else "Thinking: Off"
display(HTML(f"""
<div style="background:#1e3a1e;padding:10px;border-radius:5px;margin:10px 0;color:#d4d4d4;">
<b>Configuration Applied</b><br>
Model: {model_dropdown.value} | Region: Sydney | Temp: {CONFIG.temperature}<br>
{thinking_str} | Budget: ${CONFIG.session_cost_limit:.0f} (adjust via slider in UI)<br>
Use <code>/skills</code> to list available skills, <code>/skill use &lt;name&gt;</code> to activate
</div>
"""))

# Launch the chat interface
create_chat_ui()

---

## Quick Reminder

**Buttons:** Send | Stop | Clear (reset chat, saves memory) | Compact (shrink context) | Clean (delete disk artifacts, safe)

**Skills** — activate with `/command` or `/skill use <name>`. Keyword auto-trigger is OFF by default (`CONFIG.enable_skill_auto_trigger = False`). Detail + examples in `USER_GUIDE.md`.

| Command | Use when |
|---------|----------|
| `/verify` | Adversarial break-test recent changes |
| `/simplify` | Check reuse / quality / efficiency |
| `/done quick` | Pre-ship gate (simplify + verify) |
| `/review` | PR-style code review |
| `/security-review` | OWASP / injection / auth audit |
| `/batch` | Bulk multi-file edits in parallel |
| `/design` | 2-3 options with tradeoffs first |
| `/reflexion` | (V4.9.3) 3-pass critique-refine-judge for high-stakes outputs |
| `/clara-review` | ClaRA 5-phase production check |
| `/powerbi` | Power BI .pbip generation |
| "create a report" | Word / PDF with charts |

**Skill management:** `/skills` (list) | `/skill use <name>` | `/skill clear` | `/unskill <name>` (V4.9.1, sticky)

**Self-patching skills (V4.9.5, opt-in):** when `CONFIG.enable_skill_patching = True`, the agent proposes SKILL.md improvements after the same correction 3+ times. Review with `/skill suggestions`, apply via `/skill apply <name> --yes` (preview diff first), reject via `/skill reject <name>`. Audit log at `audit_logs/skill_patches.jsonl`. See `USER_GUIDE.md` "Self-patching skills" section for the full flow.

**Common slash commands:** `/skills` `/cost` `/context` `/status` `/revert <file>` `/compact` `/checkpoint create <name>` `/regression`

**Notebook editing:** when modifying an existing `.ipynb`, the agent should use `notebook_edit` for one-cell insert / replace / delete instead of regenerating the whole notebook.

**Memory:** auto-written to `memory.md` (4 types: user / feedback / project / reference). **Status:** `AGENT_STATUS.md` is loaded each top-level run for long tasks.

**Git:** local git tree only in SageMaker. Use status/diff/log/worktree/local commits; do not expect GitHub, `gh`, PR creation, push, pull, fetch, or clone.

**Full docs:** `USER_GUIDE.md`
